#### Signal Collection & Explicit Ratings

#### Sparsity Formula
$$\text{Sparsity(Density)} = \left(\frac{N_{\text{ratings}}}{N_{\text{users}} \times N_{\text{movies}}} \right) \times 100$$

Sparsity is the percentage of "empty" or "latent" cells in the matrix. It represents the information gap the Collaborative Filtering (CF) model must overcome.
$$\text{Sparsity} = \left( 1 - \frac{N_{\text{ratings}}}{N_{\text{users}} \times N_{\text{movies}}} \right) \times 100$$

- However, when the matrix is extremely sparse (as is typical in recommendation datasets like MovieLens), the number of actual ratings is very small compared to total possible entries.

- The difference between the two formulas is negligible (on the order of 0.0001–0.001%).

In [2]:
# Load and inspect ratings data for CF

import pandas as pd
import numpy as np
from scipy.sparse import csr_matrix
import pickle
from pathlib import Path

print("=" * 40)
print("SECTION 9: COLLABORATIVE FILTERING - DATA PREPARATION")
print("=" * 40)

# Load clean ratings
ratings = pd.read_csv("../data/processed/ratings_clean.csv")
print(f"\nRating shape: {ratings.shape}")
print(f"Columns: {ratings.columns.tolist()}")


# Basic Statistic
n_users = ratings["userId"].nunique()
n_movies = ratings["tmdbId"].nunique()
n_ratings = len(ratings)

print(f"\nUsers: {n_users:,}")
print(f"\nMovies: {n_movies:,}")
print(f"\nRatings: {n_ratings:,}")
# Sparsity = percentage of missing ratings
# Approximation valid for highly sparse matrices (difference negligible)
# Strict version: sparsity = (1 - (n_ratings / (n_users * n_movies))) * 100
print(f"Sparsity: {n_ratings / (n_users * n_movies) * 100:.4f}%")

SECTION 9: COLLABORATIVE FILTERING - DATA PREPARATION

Rating shape: (25981438, 6)
Columns: ['userId', 'movieId', 'rating', 'timestamp', 'tmdbId', 'year']

Users: 270,882

Movies: 44,702

Ratings: 25,981,438
Sparsity: 0.2146%


- The true sparsity is 99.7854% (meaning 99.7854% of possible user-movie pairs have no rating).
- The approximation 0.2146% is the proportion of filled entries — it is the inverse of the missing proportion.

##### Matrix Interaction & Sparsity Analysis
1. Theoretical Maximum Interactions (Total Search Space):
    - Formula: $N_{\text{users}} \times N_{\text{movies}}$
    - Calculation: $270,888 \times 44,702 \approx 12,109,235,376$ (12.11 Billion)
2. Observed Interactions (Actual Ratings):
    - Metric: $N_{\text{ratings}}$
    - Total: $25,981,438$ (~26 Million)
3. Matrix Density & Sparsity Results:
    - Density (Fill Rate):
    - $$\frac{25,981,438}{12,109,235,376} \times 100 \approx \mathbf{0.2146\%}$$
4. Sparsity (Missing Data):
    - $$100\% - 0.2146\% \approx \mathbf{99.7854\%}$$

#### Interpretation:
With a density of 0.2146%, the dataset exceeds the minimum threshold (0.1%) required for a stable Collaborative Filtering model. The system possesses sufficient interaction "connective tissue" to proceed with SVD++ Matrix Factorization.


### User Mean-Centering (also known as Mean-Normalization)

#### The Formula
$$r_{ui}^* = r_{ui} - \bar{r}_u$$

Where:

- $r_{ui}^*$: The Mean-Centered Rating (the result).
- $r_{ui}$: The Original Rating given by user $u$ to movie $i$ (e.g., 4 stars).
- $\bar{r}_u$: The Average Rating of all movies rated by user $u$.

In [3]:
# User mean-centering - VECTORIZED (remove optimist/pessimist bias)

print("\n" + "=" * 60)
print("STEP 9.2: USER MEAN-CENTERING (VECTORIZED)")
print("=" * 60)

# Compute user means (fast, groupby returns small DataFrame)
user_means = ratings.groupby('userId')['rating'].mean().rename('user_mean')

# Merge means back (vectorized operation)
ratings = ratings.merge(user_means, left_on='userId', right_index=True)

# Vectorized centering (NO apply, NO lambda)
ratings['rating_centered'] = ratings['rating'] - ratings['user_mean']

# Drop the temporary column
ratings = ratings.drop('user_mean', axis=1)

print(f"Centered ratings created for {len(ratings):,} rows")
print("\nCentered rating statistics:")
print(ratings['rating_centered'].describe())


STEP 9.2: USER MEAN-CENTERING (VECTORIZED)
Centered ratings created for 25,981,438 rows

Centered rating statistics:
count    2.598144e+07
mean    -3.199727e-18
std      9.529301e-01
min     -4.490079e+00
25%     -5.333333e-01
50%      9.375000e-02
75%      6.400990e-01
max      4.285714e+00
Name: rating_centered, dtype: float64


In [4]:
# Double-centering - VECTORIZED

print("\n" + "=" * 60)
print("DOUBLE-CENTERING (VECTORIZED)")
print("=" * 60)

# Compute item means of centered ratings
item_means = ratings.groupby('tmdbId')['rating_centered'].mean().rename('item_mean')

# Merge and vectorized operation
ratings = ratings.merge(item_means, left_on='tmdbId', right_index=True)
ratings['rating_double_centered'] = ratings['rating_centered'] - ratings['item_mean']
ratings = ratings.drop('item_mean', axis=1)

print("Double-centered ratings statistics:")
print(ratings['rating_double_centered'].describe())


DOUBLE-CENTERING (VECTORIZED)
Double-centered ratings statistics:
count    2.598144e+07
mean    -1.801583e-17
std      8.722091e-01
min     -4.981469e+00
25%     -4.806821e-01
50%      6.404592e-02
75%      5.663130e-01
max      5.248576e+00
Name: rating_double_centered, dtype: float64


In [5]:
# SAVE CENTERED RATINGS FOR CF TRAINING

print("\n" + "=" * 60)
print("SAVING PROCESSED RATINGS FOR CF")
print("=" * 60)

# Save full DataFrame with all columns
output_path = '../data/processed/ratings_centered.csv'
ratings.to_csv(output_path, index=False)
print(f"✓ Saved: {output_path}")
print(f"  Shape: {ratings.shape}")
print(f"  Memory: {ratings.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

# Also save a lighter version with just what's needed for Surprise
surprise_data = ratings[['userId', 'tmdbId', 'rating_double_centered']].copy()
surprise_data.columns = ['userId', 'tmdbId', 'rating']  # Rename for Surprise

surprise_path = '../data/processed/ratings_for_surprise.csv'
surprise_data.to_csv(surprise_path, index=False)
print(f"\n✓ Saved: {surprise_path} (for Surprise library)")
print(f"  Shape: {surprise_data.shape}")

# Quick verification
print("\n" + "=" * 60)
print("VERIFICATION")
print("=" * 60)
print(f"Original ratings: {len(ratings):,}")
print(f"Centered ratings mean: {ratings['rating_centered'].mean():.10f}")
print(f"Double-centered mean: {ratings['rating_double_centered'].mean():.10f}")
print(f"Sample user (ID {ratings['userId'].iloc[0]}):")
print(f"  Original: {ratings['rating'].iloc[0]}")
print(f"  Centered: {ratings['rating_centered'].iloc[0]:.3f}")
print(f"  Double-centered: {ratings['rating_double_centered'].iloc[0]:.3f}")


SAVING PROCESSED RATINGS FOR CF
✓ Saved: ../data/processed/ratings_centered.csv
  Shape: (25981438, 8)
  Memory: 3270.67 MB

✓ Saved: ../data/processed/ratings_for_surprise.csv (for Surprise library)
  Shape: (25981438, 3)

VERIFICATION
Original ratings: 25,981,438
Centered ratings mean: -0.0000000000
Double-centered mean: -0.0000000000
Sample user (ID 38150):
  Original: 4.0
  Centered: 0.226
  Double-centered: -0.103


In [6]:
ratings.head(1)

,userId,movieId,rating,timestamp,tmdbId,year,rating_centered,rating_double_centered
0,38150,1176,4.0,1995-01-09 11:46:44,1600,1995,0.226087,-0.103286


In [ ]:
# After mapping
ratings = ratings.dropna(subset=['movie_idx']).copy()  # Remove unmatched movies
print(f"Ratings after excluding unmatched movies: {len(ratings):,}")

# Use float32 for matrix (saves ~50% memory)
R = csr_matrix(
    (
        ratings['rating_double_centered'].astype(np.float32),
        (ratings['user_idx'].astype(np.int32), ratings['movie_idx'].astype(np.int32))
    ),
    shape=(len(user_mapper), len(movie_id_to_idx)),
    dtype=np.float32
)